#### 1,Download observation data; 2,Resample to CWRF grid; 3,Reasmple to monthly; 4 Generate the monthly anomaly.

In [1]:
import sys,os
import datetime 
import requests
import subprocess
import xarray      as xr
import xesmf       as xe
print('The library has loaded!')

The library has loaded!


#### Download the data, and resample to CWRF grid.

In [7]:
# Give basic informations about variable names, path of the file, and so on.
# Modify the begin and end month if necessary, as well as years and the file directory.
# years    = range(2012,2023+1)
# years    = range(1981,2023+1)
years    = range(2024,2024+1)
BegMonth,BegDay,EndMonth, EndDay = 1,1,12,31
# The folder for saving the data
data_dir = '/Volumes/ssd3/DATA/OBS/'
# The path of weight file, used to re-grid the data
stic_dir = '/Volumes/ssd3/DATA/static/'
# The observation link, do not change.
link_OBS = 'http://www.northwestknowledge.net/metdata/data/'
# The time format. follow the agreement of DAWN.
TIMEFMT  = '%Y-%m-%d-%H'

# Add more variables if necessary. 
# var_names= ['PRAVG','T2MAX','T2MIN','ASWDNS']
# var_names= ['T2MAX','T2MIN']
# var_names= ['PRAVG','ASWDNS']
var_names= ['WD','WS','VPD']
varname_mapping = {
    "PRAVG": {
        "filename_fragment" :  "pr",
        "original_var_name" :  "precipitation_amount",
    },
    "T2MAX": {
        "filename_fragment" :  "tmmx",
        "original_var_name" :  "air_temperature",
    },
    "T2MIN": {
        "filename_fragment" :  "tmmn",
        "original_var_name" :  "air_temperature",
    },
    "ASWDNS": {
        "filename_fragment" :  "srad",
        "original_var_name" :  "surface_downwelling_shortwave_flux_in_air",
    },
    "WD": {
        "filename_fragment" :  "th",
        "original_var_name" :  "wind_from_direction",
    },
    "WS": {
        "filename_fragment" :  "vs",
        "original_var_name" :  "wind_speed",
    },
    "VPD": {
        "filename_fragment" :  "vpd",
        "original_var_name" :  "mean_vapor_pressure_deficit",
    },
}


# Define The functions used for download and regrid.
def regrid_observation(stic_dir,data_dir,file_name_raw,file_name_obs):
    # Read in the meteorology data on a regular lat/lon grid
    wtg_file = stic_dir + 'OBS2CWRF_weights_file.nc'
    # Read in the meteorology data on a regular lat/lon grid
    ds_in = xr.open_dataset(data_dir + 'raw_daily/' + file_name_raw)
    # Read in the CWRF grid file
    ds_cwrf  = xr.open_dataset( stic_dir + 'geo_em.d01_30.nc')
    # Create a new dataset with the latitude and longitude from the WRF grid
    ds_out = xr.Dataset({'lat': ds_cwrf['XLAT_M'].isel(Time=0), 'lon': ds_cwrf['XLONG_M'].isel(Time=0)})
    # Create the regridder using the pre-generated weights
    regridder = xe.Regridder(ds_in, ds_out, method='bilinear', filename=wtg_file, reuse_weights=True)
    # Regrid the meteorology data
    ds_in_regridded = regridder(ds_in)
    # Modify the variable name and coordinate name 
    ds_in_regridded = ds_in_regridded.rename({varname_mapping[var_name]['original_var_name']  : var_name , 'day'  : 'time' })
    # Export the result to netcdf format.
    ds_in_regridded.to_netcdf(data_dir + 'regrid_daily/'+ file_name_obs)
    print(f'the data {file_name_obs} has download and regrid.')
    
def download_regrid_observation(link_OBS,stic_dir,data_dir,file_name_raw,file_name_obs):
    # Check if the file exists on internet.
    file_exists = requests.get(f'{link_OBS}{file_name_original}').status_code == 200
    if file_exists:
        # Download the dataset, and rename it.
        os.system(f'wget {link_OBS     +      file_name_original} -P {data_dir}raw_daily/  >& log.txt')
        os.system(f'mv   {data_dir}raw_daily/{file_name_original}    {data_dir}raw_daily/{file_name_raw}')
        # Interpolate the raw data onto the DAWN grid
        regrid_observation(stic_dir,data_dir,file_name_raw,file_name_obs)
    else:
        print(f'File not exit :  {file_name_original} ')


#### Download and interpolate the observation to DAWN grid.

In [8]:
for var_name in var_names:
    for year in years:
        # Generate the file names which follows Dawn agreement.
        # set time_beg and time_end for trim the data through time dimension
        time_beg = datetime.datetime(year, BegMonth,  BegDay,  0).strftime(TIMEFMT)
        time_end = datetime.datetime(year, EndMonth,  EndDay, 18).strftime(TIMEFMT)
        # Assemble the filenames
        file_name_original = varname_mapping[var_name]['filename_fragment'] + "_" + str(year) + '.nc'
        file_name_raw      = f'OBS_raw_{var_name}_{year}.nc'
        file_name_obs      = f'OBS_{var_name}_{time_beg}_{time_end}.nc'
        # Download the file.
        if os.path.exists(f'{data_dir}raw_daily/{file_name_raw}'):               # raw file on the disk
            print(f"File already exists: {data_dir}raw_daily/{file_name_raw}")
            if os.path.exists(f'{data_dir}{file_name_obs}'):                     # regrided file on the disk
                print(f"File already exists: {data_dir}{file_name_obs}")
            else:                                                                # if not, regrid from raw file.
                regrid_observation(stic_dir,data_dir,file_name_raw,file_name_obs)
        else:                                                                    # raw file not exists, download and interpolate.
            download_regrid_observation(link_OBS,stic_dir,data_dir,file_name_raw,file_name_obs)


the data OBS_PRAVG_2024-01-01-00_2024-12-31-18.nc has download and regrid.
the data OBS_ASWDNS_2024-01-01-00_2024-12-31-18.nc has download and regrid.


#### Resample to monthly data

In [9]:
for var_name in var_names:
    for year in years:
        # Generate the input and output file name
        time_beg = datetime.datetime(year, BegMonth,  BegDay,  0).strftime(TIMEFMT)
        time_end = datetime.datetime(year, EndMonth,  EndDay, 18).strftime(TIMEFMT)
        file_name_obs      = f'OBS_{var_name}_{time_beg}_{time_end}.nc'
        file_name_monthly  = f'OBS_monthly_{var_name}_{time_beg}_{time_end}.nc'
        # Read,Resample, and Save the data
        ds_day   = xr.open_dataset(f'{data_dir}regrid_daily/{file_name_obs}')
        ds_mnth  = ds_day.resample(time='M').mean()
        ds_mnth.to_netcdf(f'{data_dir}regrid_monthly/{file_name_monthly}')
        print(f'{file_name_monthly} has resampled and saved.')

OBS_monthly_PRAVG_2024-01-01-00_2024-12-31-18.nc has resampled and saved.
OBS_monthly_ASWDNS_2024-01-01-00_2024-12-31-18.nc has resampled and saved.


#### Calculate the anomaly. take 2012-2023 as climatology.

In [10]:
for var_name  in var_names:
    file_list = []
    for year in years:
        filename = f'{data_dir}regrid_monthly/OBS_monthly_{var_name}_{year}-01-01-00_{year}-12-31-18.nc'
        if os.path.exists(filename):
            file_list.append(filename)

    # Read multiple data
    ds_mnth   = xr.open_mfdataset(file_list, combine='by_coords',chunks={'south_north': 23})
    # Calculate the anomaly
    gb        = ds_mnth.groupby('time.month')
    anomaly   = gb - gb.mean('time')
    # Drop redundant coords.
    anomaly   = anomaly.squeeze('crs',drop=True)
    anomaly   = anomaly.reset_coords('month',drop=True)
    # Save the anomaly data
    anomaly.to_netcdf(f'{data_dir}regrid_monthly_anomaly/OBS_monthly_anomaly_{var_name}_{years[0]}-{years[-1]}.nc')

In [4]:
data_dir = '/Volumes/ssd3/DATA/OBS/'
ds = xr.open_dataset(f'{data_dir}try/vpd_1979.nc')
ds

<xarray.Dataset>
Dimensions:                      (lon: 1386, lat: 585, day: 365, crs: 1)
Coordinates:
  * lon                          (lon) float64 -124.8 -124.7 ... -67.1 -67.06
  * lat                          (lat) float64 49.4 49.36 49.32 ... 25.11 25.07
  * day                          (day) datetime64[ns] 1979-01-01 ... 1979-12-31
  * crs                          (crs) uint16 3
Data variables:
    mean_vapor_pressure_deficit  (day, lat, lon) float32 ...
Attributes: (12/19)
    geospatial_bounds_crs:      EPSG:4326
    Conventions:                CF-1.6
    geospatial_bounds:          POLYGON((-124.7666666333333 49.40000000000000...
    geospatial_lat_min:         25.066666666666666
    geospatial_lat_max:         49.40000000000000
    geospatial_lon_min:         -124.7666666333333
    ...                         ...
    date:                       02 July 2019
    note1:                      The projection information for this file is: ...
    note2:                      Citation: Abatzoglou, J.T., 2013, Development...
    note3:                      Data in slices after last_permanent_slice (1-...
    note4:                      Data in slices after last_provisional_slice (...
    note5:                      Days correspond approximately to calendar day...